## Pixels to Predictions — `final_project.ipynb`

**Course submission pipeline (SmolVLM + LoRA/DoRA, ≤5M trainable params).**  
Full step-by-step reproduction, checkpoint paths, and grading-oriented detail are in the repository **`README.md`** — read that file first.

---

### End-to-end pipeline (what this notebook does)

| Step | Description |
|------|-------------|
| 1 | **Data**: Auto-find `DATA_DIR` (or `DATA_DIR_OVERRIDE`) → load `train` / `val` / `test` CSVs and parse `choices`. |
| 2 | **Model**: Load **`HuggingFaceTB/SmolVLM-500M-Instruct`** (`AutoModelForImageTextToText`), FP16 on GPU. |
| 3 | **Adapter**: PEFT **LoRA or DoRA** on LLM layers (`PREFERRED_LORA_SUFFIXES`); rank `r` chosen so **trainable params ≤ `MAX_TRAINABLE_PARAMS`**. |
| 4 | **Training** *(if `RUN_TRAINING=True`)* | Masked LM loss on **answer letter** tokens only; AdamW + cosine schedule + warmup; optional **train∪val** (`USE_VAL_FOR_TRAINING`). |
| 5 | **Checkpoints** | Save adapter + processor to **`ADAPTER_DIR`** (`adapter_config.json`, weights). Optional **`CAPTION_CACHE_PATH`** if captions enabled. |
| 6 | **Inference** | Per-option **log-likelihood** (letter + ensemble detail weight) → argmax → **`submission.csv`** (`id`, `answer`). |

---

### Reproducibility

- **Seed**: `SEED = 42` — applied to `torch`, `numpy`, `random`, and CUDA (see code cell below).
- **Training vs inference**:
  - **Train + infer**: `RUN_TRAINING=True` → trains, saves checkpoint, then predicts test.
  - **Infer only**: `RUN_TRAINING=False` **and** existing `adapter_config.json` under `ADAPTER_DIR` → loads adapter, skips training.

---

### Checkpoints (artifacts)

| Variable | Typical path (Kaggle) | Purpose |
|----------|------------------------|---------|
| `ADAPTER_DIR` | `/kaggle/working/smolvlm-mcqa-lora/` | **PEFT adapter** — reload for inference-only runs |
| `CAPTION_CACHE_PATH` | `/kaggle/working/caption_cache.json` | Cached image captions when `USE_IMAGE_CAPTION=True` |
| Output file | `/kaggle/working/submission.csv` | Competition submission |

---

### Kaggle quick start

1. Attach **competition data** to the notebook.
2. **Settings → Accelerator → GPU T4 ×2 → Save → Restart session**.
3. Run **Save & Run All** on this notebook (optional: `!pip install -q "peft>=0.13.0"` if import fails).

All hyperparameters live in the **`CONFIG`** section of the next code cell (`IMG_SIZE`, `TRAIN_EPOCHS`, `TRY_DORA`, etc.).

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# Do NOT print every file under /kaggle/input here — it floods Kaggle "Logs" (10k+ lines) on
# Save Version / submit runs and hides real progress. Uncomment only when debugging paths interactively.
# for dirname, _, filenames in os.walk("/kaggle/input"):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))
if os.path.isdir("/kaggle/input"):
    print("[Input] /kaggle/input mounted (full listing skipped in this cell).")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### Implementation cell (below)

The next code cell contains **all executable logic**: imports, **`CONFIG`** (hyperparameters), **`discover_data_dir`**, LoRA/DoRA probe, optional **training loop**, checkpoint **`save_pretrained`**, optional **`merge_and_unload`**, and **test inference** writing **`submission.csv`**.

Do not reorder cells when reproducing; run **top → bottom**.

In [2]:
# =============================================================================
# Pixels to Predictions — MCQA with SmolVLM + PEFT (LoRA/DoRA, ≤5M trainable)
# =============================================================================
# Documentation: see README.md for reproducibility, checkpoint paths, Kaggle steps.
# Markdown cells above summarize the pipeline for reviewers.
# Optional dependency (uncomment on Kaggle if ImportError):
# !pip install -q "peft>=0.13.0" "torchvision"

import json
import math
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageEnhance
from transformers import AutoModelForImageTextToText, AutoProcessor, get_cosine_schedule_with_warmup

try:
    from peft import LoraConfig, PeftModel, get_peft_model, TaskType
except ImportError as e:
    raise ImportError("pip install 'peft>=0.13.0'") from e

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None


# =============================================================================
# CONFIGURATION — balance speed vs score (tuned to cut wall time without big LB hits).
# Tighter cap: keep IMG_SIZE=384, TRAIN_EPOCHS 3, subsampled val + every-2nd-epoch val.
# =============================================================================
# If CUDA OOM: lower IMG_SIZE → USE_IMAGE_CAPTION False → fewer TRAIN_EPOCHS → TRY_DORA False.

# Absolute path to folder containing train.csv (None = auto-detect).
DATA_DIR_OVERRIDE = None

# If True: attach LoRA/DoRA and run training, then save adapter and run test inference.
# If False: load existing adapter from ADAPTER_DIR (must contain adapter_config.json).
RUN_TRAINING = True  # Set False to load existing adapter from ADAPTER_DIR and skip training.

# Where PEFT saves adapter weights + tokenizer files (persists under /kaggle/working on Kaggle).
ADAPTER_DIR = Path("/kaggle/working/smolvlm-mcqa-lora") if Path("/kaggle").exists() else Path("./smolvlm-mcqa-lora")
# JSON map image_path → caption string when USE_IMAGE_CAPTION is enabled.
CAPTION_CACHE_PATH = Path("/kaggle/working/caption_cache.json") if Path("/kaggle").exists() else Path("./caption_cache.json")

# Competition rule: total trainable parameters must not exceed this count.
MAX_TRAINABLE_PARAMS = 5_000_000
# Prefer Weight-Decomposed LoRA (DoRA); falls back to plain LoRA if PEFT rejects use_dora.
TRY_DORA = True

# LoRA injects into these module name suffixes inside the language model (not vision tower).
# Attention + MLP gives stronger adaptation per rank cap than attention-only.
PREFERRED_LORA_SUFFIXES = ("q_proj", "v_proj", "gate_proj", "up_proj", "down_proj")

# Optimization hyperparameters (effective batch size = 1 × GRADIENT_ACCUMULATION_STEPS).
TRAIN_EPOCHS = 3  # Fewer epochs for time; raise to 4–5 if score plateaus and you have GPU hours.
LEARNING_RATE = 1.8e-4  # Slightly below 2e-4 for longer schedule (5 ep); reduces late overshoot.
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10  # A bit more warmup vs total steps when training longer.
# Subsample training rows for debugging (None = use all of train_fit_df).
MAX_TRAIN_SAMPLES = None
# Divide loss by this and step optimizer every N micro-batches to save VRAM.
GRADIENT_ACCUMULATION_STEPS = 4
# Concatenate validation rows into training (common for leaderboard; set False for strict train-only).
USE_VAL_FOR_TRAINING = True

# 384: faster train+infer; try 416/448 locally for +detail if you have time (slower).
IMG_SIZE = 384
# VLM-generated captions can help text-image alignment; very slow (one generate per image). Leave False
# for speed; set True only if you accept long precompute for a possible small LB gain.
USE_IMAGE_CAPTION = False
CAPTION_MAX_NEW = 40
# Blend score = letter_logprob_mean + WEIGHT × choice_text_logprob_mean (see predict_one).
# Sweep ~0.35–0.55; higher weights joint letter+choice scoring (often helps chart-heavy items).
ENSEMBLE_DETAIL_WEIGHT = 0.52

# --- Validation + best checkpoint ---
EVAL_VAL_EACH_EPOCH = True
SAVE_BEST_BY_VAL = True
VAL_EVAL_MAX_SAMPLES = 512  # Subsample val for speed (~full-rank correlation often OK); None = full val (slow).
EVAL_VAL_EVERY_N_EPOCHS = 2  # Val every 2nd epoch + always last epoch (halves val passes).
# If USE_VAL_FOR_TRAINING, val is also in training — val acc is optimistic.

# --- Inference (batching = fewer GPU launches, same math) ---
INFER_SCORE_MODE = "ensemble"  # "ensemble" | "letter_only"
USE_BATCHED_OPTION_SCORING = True

# Prompt tail must stay identical between training labels and inference scoring.
ANSWER_LINE_STYLE = "Answer:"
PROMPT_INSTRUCTION = "Select exactly one letter (A, B, C, …) for the best choice."

# Random flip / color jitter on training images only (not at test time).
TRAIN_AUGMENT = True

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def discover_data_dir() -> Path:
    """Locate the dataset root directory that contains ``train.csv``.

    Search order: explicit override → env vars → parents of cwd → Kaggle /kaggle/input
    (including ``competitions/<slug>``) → local ../input paths → legacy fixed paths.

    Returns:
        pathlib.Path: Resolved directory containing ``train.csv``.

    Raises:
        FileNotFoundError: If no valid data root is found.
    """

    def ok(p: Path) -> bool:
        """Return True if ``p / "train.csv"`` exists — quick validity check for a data root."""
        return (p / "train.csv").is_file()

    if DATA_DIR_OVERRIDE is not None:
        p = Path(DATA_DIR_OVERRIDE).expanduser().resolve()
        if ok(p):
            return p
        raise FileNotFoundError(f"DATA_DIR_OVERRIDE invalid: {p}")

    for key in ("PIXELS_DATA_DIR", "KAGGLE_DATA_DIR"):
        raw = os.environ.get(key)
        if raw:
            p = Path(raw).expanduser().resolve()
            if ok(p):
                return p

    here = Path.cwd().resolve()
    for _ in range(6):
        if ok(here):
            return here
        here = here.parent

    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        comp = kaggle / "competitions"
        if comp.is_dir():
            for slug in ("pixels-to-predictions", "pixels-to-predictions-dl-vision-challenge"):
                p = comp / slug
                if ok(p):
                    return p
            for child in sorted(comp.iterdir()):
                if ok(child):
                    return child
        for slug in ("pixels-to-predictions-dl-vision-challenge", "pixels-to-predictions"):
            p = kaggle / slug
            if ok(p):
                return p
        for child in sorted(kaggle.iterdir()):
            if ok(child):
                return child

    for rel in (
        Path("../input/pixels-to-predictions"),
        Path("../input/pixels-to-predictions-dl-vision-challenge"),
    ):
        p = rel.resolve()
        if ok(p):
            return p

    for legacy in (
        Path("/kaggle/input/competitions/pixels-to-predictions"),
        Path("/kaggle/input/competitions/pixels-to-predictions-dl-vision-challenge"),
    ):
        if ok(legacy):
            return legacy

    raise FileNotFoundError("Could not find train.csv — set DATA_DIR_OVERRIDE.")


DATA_DIR = discover_data_dir()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# TF32 tensor cores on Ampere-class GPUs (e.g. T4): faster matmul, negligible memory impact.
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.benchmark = True  # fixed IMG_SIZE: can speed conv/attn after a short warmup

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count(), "| DATA_DIR:", DATA_DIR)

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

for df in (train_df, val_df, test_df):
    df["choices"] = df["choices"].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

train_fit_df = pd.concat([train_df, val_df], ignore_index=True) if USE_VAL_FOR_TRAINING else train_df
print("Training rows:", len(train_fit_df), "| USE_VAL_FOR_TRAINING:", USE_VAL_FOR_TRAINING)


def image_path(row) -> Path:
    """Resolve the on-disk image file for one dataframe row.

    Args:
        row: A row (e.g. ``pd.Series``) that includes the string column ``image_path``
            relative to ``DATA_DIR``.

    Returns:
        pathlib.Path: Absolute path to the image file. Prefer ``DATA_DIR / image_path``;
            if that file is missing but the CSV uses ``images/...``, also try the nested
            layout ``images/images/...`` (some local unzip trees duplicate the folder name).

    Note:
        Callers should handle ``FileNotFoundError`` if neither location exists.
    """
    rel = str(row["image_path"]).replace("\\", "/")
    primary = DATA_DIR / rel
    if primary.is_file():
        return primary
    if rel.startswith("images/"):
        nested = DATA_DIR / rel.replace("images/", "images/images/", 1)
        if nested.is_file():
            return nested
    return primary


def augment_pil_train(img: Image.Image) -> Image.Image:
    """Resize and optionally augment a PIL image for the training loop only.

    Args:
        img: RGB ``PIL.Image`` opened from disk (any size).

    Returns:
        PIL.Image.Image: Square ``IMG_SIZE × IMG_SIZE`` image. If ``TRAIN_AUGMENT`` is
        True, applies random horizontal flip and mild brightness/contrast jitter; otherwise
        deterministic resize only (matches eval-style preprocessing shape).

    Note:
        Inference and caption precompute use separate resize paths — keep ``IMG_SIZE``
        consistent everywhere.
    """
    if not TRAIN_AUGMENT:
        return img.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    img = ImageEnhance.Brightness(img).enhance(1.0 + 0.1 * (random.random() - 0.5))
    img = ImageEnhance.Contrast(img).enhance(1.0 + 0.08 * (random.random() - 0.5))
    return img


def build_prompt(row: pd.Series, caption=None) -> str:
    """Build the text half of a SmolVLM sample; ends at ``ANSWER_LINE_STYLE`` with no answer token yet.

    Args:
        row: Training/validation/test row with ``question``, ``choices`` (list), and optional
            ``lecture``, ``hint``, ``grade``, ``subject``, ``topic``, ``category``.
        caption: Optional short image description string when ``USE_IMAGE_CAPTION`` is enabled.

    Returns:
        str: Prompt beginning with the ``<image>`` placeholder, optional caption and metadata,
        question, enumerated choices, instruction line, and finally ``Answer:`` (no space after colon).
        Training concatenates a space plus the gold letter after this string (same tail as MCQA scoring).

    Note:
        Keep ``ANSWER_LINE_STYLE`` / ``PROMPT_INSTRUCTION`` aligned between train and test or
        likelihood scores will not match the supervised distribution.
    """
    meta_lines = []
    for key in ("grade", "subject", "topic", "category"):
        v = row.get(key)
        if pd.notna(v) and str(v).strip():
            meta_lines.append(f"{key}: {str(v).strip()}")
    meta = ("Background:\n" + "\n".join(meta_lines) + "\n\n") if meta_lines else ""

    cap_block = ""
    if caption and str(caption).strip():
        cap_block = "Image description:\n" + str(caption).strip() + "\n\n"

    parts = []
    lec = row.get("lecture", "")
    h = row.get("hint", "")
    if pd.notna(lec) and str(lec).strip():
        parts.append(str(lec).strip())
    if pd.notna(h) and str(h).strip():
        parts.append(str(h).strip())
    ctx = "\n".join(parts)

    choices = row["choices"]
    lines = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    prompt = "<image>\n"
    prompt += cap_block
    prompt += meta
    if ctx:
        prompt += f"Context:\n{ctx}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{lines}\n"
    prompt += PROMPT_INSTRUCTION + "\n"
    prompt += ANSWER_LINE_STYLE.rstrip()
    return prompt


def infer_lora_targets(model: torch.nn.Module, preferred=None):
    """Discover ``target_modules`` suffix strings for PEFT LoRA/DoRA on the language model.

    Walks ``model.named_modules()``, keeps ``nn.Linear`` layers whose names look like LLM
    projections (``q_proj``, ``v_proj``, ``gate_proj``, etc.) under ``text_model`` /
    ``language_model``, and excludes vision tower / SigLIP-related paths.

    Args:
        model: Loaded Hugging Face model (same architecture as training).
        preferred: Optional iterable of suffix strings (e.g. ``PREFERRED_LORA_SUFFIXES``).
            If provided and non-empty matches exist, only those suffixes are returned; else
            the search is retried without filtering, then broadened to minimal ``q/v/o_proj``.

    Returns:
        list[str]: Sorted unique module suffix names for ``LoraConfig(target_modules=...)``.
    """
    suffixes = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
    skip = ("vision", "visual", "image_tower", "patch_embed", "siglip")
    hits = []
    for name, mod in model.named_modules():
        if not isinstance(mod, torch.nn.Linear):
            continue
        if not name.endswith(suffixes):
            continue
        ln = name.lower()
        if any(s in ln for s in skip):
            continue
        if "text_model" in ln or "language_model" in ln:
            suf = name.rsplit(".", 1)[-1]
            if preferred is None or suf in preferred:
                hits.append(suf)
    out = sorted(set(hits))
    if out:
        return out
    if preferred is not None:
        return infer_lora_targets(model, preferred=None)
    for name, mod in model.named_modules():
        if isinstance(mod, torch.nn.Linear) and name.endswith(("q_proj", "v_proj", "o_proj")):
            if not any(s in name.lower() for s in skip):
                hits.append(name.rsplit(".", 1)[-1])
    return sorted(set(hits)) or ["q_proj", "v_proj"]


def count_trainable(model: torch.nn.Module) -> int:
    """Count trainable parameters (adapter + any unfrozen base weights).

    Used after wrapping with PEFT to verify the competition cap ``MAX_TRAINABLE_PARAMS``.

    Args:
        model: Possibly wrapped ``PeftModel`` or bare ``nn.Module``.

    Returns:
        int: Sum of ``p.numel()`` for all ``p`` with ``p.requires_grad``.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def model_dev(model):
    """Infer which ``torch.device`` holds model parameters for ``.to()`` / autocast.

    Some wrappers expose ``model.device``; distributed or partially moved modules may only
    expose devices via ``next(model.parameters()).device``.

    Args:
        model: Hugging Face or PEFT-wrapped model used in training or inference.

    Returns:
        torch.device: CUDA device when available, else CPU.
    """
    try:
        d = model.device
        if d is not None and str(d) != "cpu":
            return d
    except Exception:
        pass
    return next(model.parameters()).device


def load_base():
    """Load ``MODEL_ID`` once with dtype and device placement suitable for fine-tuning.

    Uses global ``dtype`` (fp16 on CUDA, fp32 on CPU), ``low_cpu_mem_usage=True``, and
    ``device_map=\"auto\"`` on GPU so large checkpoints shard across devices when needed.

    Returns:
        AutoModelForImageTextToText: Fresh pretrained weights (before ``get_peft_model``).

    Note:
        Caller deletes probes after rank search to free VRAM before the real train graph.
    """
    kw = dict(
        pretrained_model_name_or_path=MODEL_ID,
        dtype=dtype,
        low_cpu_mem_usage=True,
    )
    if torch.cuda.is_available():
        kw["device_map"] = "auto"
    m = AutoModelForImageTextToText.from_pretrained(**kw)
    if not torch.cuda.is_available():
        m.to(device)
    return m


processor = AutoProcessor.from_pretrained(MODEL_ID)
if getattr(processor, "tokenizer", None) is not None and processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

mt = load_base()
target_modules = infer_lora_targets(mt, PREFERRED_LORA_SUFFIXES)
del mt
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("LoRA target_modules (preferred attention+MLP):", target_modules)


def lora_config_with_dora(r: int, alpha: int, tm: list, use_dora: bool):
    """Construct a ``peft.LoraConfig`` for causal LM, optionally with DoRA.

    Args:
        r: LoRA rank (lower rank → fewer trainable params).
        alpha: LoRA scaling (often set ≥ r; here ``max(r, 16)`` in the rank search loop).
        tm: List of ``target_modules`` suffix strings from ``infer_lora_targets``.
        use_dora: If True, passes ``use_dora=True`` when the installed PEFT supports it.

    Returns:
        LoraConfig: Ready for ``get_peft_model(model, config)``.

    Note:
        On ``TypeError`` (older PEFT), falls back to plain LoRA without DoRA.
    """
    kw = dict(
        task_type=TaskType.CAUSAL_LM,
        r=r,
        lora_alpha=alpha,
        lora_dropout=0.05,
        bias="none",
        target_modules=tm,
    )
    if use_dora:
        kw["use_dora"] = True
    try:
        return LoraConfig(**kw)
    except TypeError:
        kw.pop("use_dora", None)
        return LoraConfig(**kw)


best = None
use_dora_effective = TRY_DORA
for r in (14, 12, 10, 8, 6, 4):
    probe = load_base()
    cfg = lora_config_with_dora(r, max(r, 16), target_modules, use_dora_effective)
    try:
        probe = get_peft_model(probe, cfg)
    except Exception as e:
        print("get_peft_model failed, retry without DoRA:", e)
        use_dora_effective = False
        probe = load_base()
        cfg = lora_config_with_dora(r, max(r, 16), target_modules, False)
        probe = get_peft_model(probe, cfg)
    n = count_trainable(probe)
    del probe
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    tag = "DoRA" if use_dora_effective else "LoRA"
    print(f"  trial {tag} r={r} -> trainable {n:,}")
    if n <= MAX_TRAINABLE_PARAMS:
        best = (r, max(r, 16), use_dora_effective)
        break

if best is None:
    raise RuntimeError("Cannot fit adapters under MAX_TRAINABLE_PARAMS — reduce IMG_SIZE or targets.")

LORA_R, LORA_ALPHA, DORA_ACTIVE = best
print(f"Using {'DoRA' if DORA_ACTIVE else 'LoRA'} r={LORA_R}, lora_alpha={LORA_ALPHA}")

lora_config = lora_config_with_dora(LORA_R, LORA_ALPHA, target_modules, DORA_ACTIVE)

model = load_base()

if RUN_TRAINING:
    model = get_peft_model(model, lora_config)
    assert count_trainable(model) <= MAX_TRAINABLE_PARAMS
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
elif ADAPTER_DIR.is_dir() and (ADAPTER_DIR / "adapter_config.json").is_file():
    model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
    print("Loaded adapter from", ADAPTER_DIR)
else:
    print("No adapter — base model only.")

caption_cache: dict = {}
if CAPTION_CACHE_PATH.is_file():
    try:
        caption_cache = json.loads(CAPTION_CACHE_PATH.read_text(encoding="utf-8"))
    except Exception:
        caption_cache = {}


@torch.inference_mode()
def generate_caption(m, pil_img: Image.Image) -> str:
    """Greedy-generate a short English caption using the same VLM checkpoint.

    Prompt asks for one sentence describing the figure; decoding uses ``generate()`` with
    ``do_sample=False``. The tail after ``ANSWER_LINE_STYLE`` is returned (truncated).

    Args:
        m: Model with ``.generate`` (base or PEFT-wrapped during cache warmup).
        pil_img: RGB image already resized if desired (caption path uses ``IMG_SIZE`` resize elsewhere).

    Returns:
        str: Plain-text caption, max 512 chars, for injection into ``build_prompt`` as metadata.

    Note:
        Disabled when ``USE_IMAGE_CAPTION`` is False — avoids slow cache builds on every run.
    """
    cap_prompt = (
        "<image>\nBriefly describe this science figure, diagram, or map in one English sentence for a student.\n"
        + ANSWER_LINE_STYLE.rstrip()
    )
    inp = processor(images=[pil_img], text=[cap_prompt], return_tensors="pt")
    inp = {k: v.to(model_dev(m)) if torch.is_tensor(v) else v for k, v in inp.items()}
    gen_kw = dict(max_new_tokens=CAPTION_MAX_NEW, do_sample=False, pad_token_id=processor.tokenizer.pad_token_id)
    if model_dev(m).type == "cuda":
        with torch.autocast(device_type="cuda", dtype=dtype):
            out_ids = m.generate(**inp, **gen_kw)
    else:
        out_ids = m.generate(**inp, **gen_kw)
    text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    if ANSWER_LINE_STYLE in text:
        text = text.split(ANSWER_LINE_STYLE, 1)[-1].strip()
    return text[:512]


def ensure_caption_for_row(m, row) -> str:
    """Return a caption for ``row['image_path']``, using disk cache or on-the-fly generation.

    Args:
        m: Model used for ``generate_caption`` when the key is missing.
        row: Dataframe row with ``image_path`` and image file readable via ``image_path(row)``.

    Returns:
        str: Caption text; also writes into module-global ``caption_cache`` for reuse.

    Note:
        Pair with ``save_caption_cache()`` to persist across notebook restarts on Kaggle.
    """
    key = str(row["image_path"])
    if key in caption_cache and caption_cache[key]:
        return caption_cache[key]
    pth = image_path(row)
    pil = Image.open(pth).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
    cap = generate_caption(m, pil)
    caption_cache[key] = cap
    return cap


def save_caption_cache():
    """Write ``caption_cache`` to ``CAPTION_CACHE_PATH`` as pretty-printed UTF-8 JSON.

    Side effects:
        Overwrites the cache file on disk (indent=1 for readability in small notebooks).

    Note:
        Called after bulk warmup and when inference lazily fills missing captions.
    """
    CAPTION_CACHE_PATH.write_text(json.dumps(caption_cache, ensure_ascii=False, indent=1), encoding="utf-8")


if USE_IMAGE_CAPTION:
    print("Building/filling image caption cache (one forward each unique image)...")
    model.eval()
    seen = set()
    for df in (train_fit_df, test_df):
        for _, row in df.iterrows():
            k = str(row["image_path"])
            if k in seen:
                continue
            seen.add(k)
            if k not in caption_cache or not caption_cache[k]:
                try:
                    ensure_caption_for_row(model, row)
                except Exception as e:
                    print("caption skip", k, e)
    save_caption_cache()
    print("Caption cache entries:", len(caption_cache), "| saved:", CAPTION_CACHE_PATH)


def prefix_token_len(image, prompt_text: str) -> int:
    """Token-count of the prompt-only prefix for one (image, text) pair.

    Uses the global ``processor`` so vision tokens + text tokens match training forward.

    Args:
        image: PIL image aligned with how training tokenizes (e.g. augmented train crop).
        prompt_text: Text produced by ``build_prompt`` **without** the answer letter.

    Returns:
        int: Sequence length of ``input_ids`` for that prefix — columns ``[:, :lp]`` get label -100.

    Note:
        Must match the prefix used in ``full_text = prompt + \" \" + letter`` for correct masking.
    """
    enc = processor(images=[image], text=[prompt_text], return_tensors="pt", padding=True)
    return int(enc["input_ids"].shape[1])


def sum_token_logprobs(model, inp_f, len_prefix: int):
    """Sum log-probabilities of observed continuation tokens (for MCQA scoring).

    Runs one forward pass with ``inp_f`` (full sequence). For each position ``j >= len_prefix``,
    adds ``log p(token[j] | image, tokens[:j])`` using the causal LM head.

    Args:
        model: Evaluated model producing ``logits`` of shape ``(B, T, V)``.
        inp_f: Processor output dict for the **full** string (image + prompt + suffix), on CPU;
            tensors are moved to ``model_dev(model)`` inside this function.
        len_prefix: First index **of continuation**: tokens at indices ``len_prefix..T-1`` are scored.

    Returns:
        tuple[float, int]: ``(total_logprob, num_tokens_scored)`` for averaging in ``score_suffix``.

    Note:
        Causal LM indexing: ``logits[t]`` predicts token ``t+1``, so the logprob for token at
        index ``j`` is ``log_probs[0, j - 1, input_ids[0, j]]``.
    """
    md = model_dev(model)
    inp_f = {k: v.to(md) if torch.is_tensor(v) else v for k, v in inp_f.items()}
    with torch.inference_mode():
        if md.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=dtype):
                out = model(**inp_f)
        else:
            out = model(**inp_f)
    logits = out.logits.float()
    log_probs = F.log_softmax(logits, dim=-1)
    input_ids = inp_f["input_ids"]
    total = 0.0
    nt = 0
    for j in range(len_prefix, input_ids.shape[1]):
        tid = input_ids[0, j]
        total += log_probs[0, j - 1, tid].item()
        nt += 1
    return total, nt


def sum_token_logprobs_batch(model, inp_f, len_prefix: int):
    """Mean continuation log-prob per row for a padded batch (shared ``len_prefix``)."""
    md = model_dev(model)
    inp_f = {k: v.to(md) if torch.is_tensor(v) else v for k, v in inp_f.items()}
    with torch.inference_mode():
        if md.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=dtype):
                out = model(**inp_f)
        else:
            out = model(**inp_f)
    logits = out.logits.float()
    log_probs = F.log_softmax(logits, dim=-1)
    input_ids = inp_f["input_ids"]
    attn = inp_f.get("attention_mask")
    if attn is None:
        attn = torch.ones_like(input_ids)
    B, T = input_ids.shape
    out_scores = []
    for b in range(B):
        seq_len = int(attn[b].sum().item())
        total = 0.0
        nt = 0
        for j in range(len_prefix, seq_len):
            tid = input_ids[b, j]
            total += log_probs[b, j - 1, tid].item()
            nt += 1
        out_scores.append(total / max(nt, 1))
    return out_scores


def align_prefix_len(inp_p, inp_f) -> int:
    """Length of the longest shared token prefix between prompt-only and full encodings.

    Used so ``sum_token_logprobs`` only scores tokens belonging to ``suffix``, even when the
    processor emits slightly different splits for ``prompt`` vs ``prompt+suffix``.

    Args:
        inp_p: Batch encoding of ``(image, prompt)`` only.
        inp_f: Batch encoding of ``(image, prompt + suffix)``.

    Returns:
        int: Number of matching leading token ids; 0 forces fallback to ``prefix_token_len``.
    """
    ids_p = inp_p["input_ids"][0]
    ids_f = inp_f["input_ids"][0]
    if ids_f.shape[0] <= ids_p.shape[0]:
        return 0
    if torch.equal(ids_f[: ids_p.shape[0]], ids_p):
        return int(ids_p.shape[0])
    k = 0
    for a, b in zip(ids_p.tolist(), ids_f.tolist()):
        if a == b:
            k += 1
        else:
            break
    return k


def score_suffix(model, processor, image: Image.Image, prompt: str, suffix: str) -> float:
    """Average per-token log-probability of ``suffix`` conditioned on (image, prompt).

    Encodes ``prompt`` and ``prompt + suffix``, aligns prefix lengths, then divides total
    logprob by the number of continuation tokens (numeric stability vs sequence length).

    Args:
        model: Model in ``eval`` mode.
        processor: Same ``AutoProcessor`` as training (must match tokenization).
        image: Eval-time resized PIL image (``IMG_SIZE`` square, no train augmentation).
        prompt: Output of ``build_prompt`` without answer.
        suffix: Hypothesis continuation, e.g. ``\" A\"`` or ``\" A. choice text\"``.

    Returns:
        float: Mean log-probability over scored tokens (higher is better for argmax).
    """
    full_text = prompt + suffix
    inp_p = processor(images=[image], text=[prompt], return_tensors="pt")
    inp_f = processor(images=[image], text=[full_text], return_tensors="pt")
    lp = align_prefix_len(inp_p, inp_f)
    if lp == 0:
        lp = prefix_token_len(image, prompt)
    s, nt = sum_token_logprobs(model, inp_f, lp)
    return s / max(nt, 1)


def score_suffixes_batch(model, processor, image: Image.Image, prompt: str, suffixes: list) -> list:
    """Batched ``score_suffix`` for one image and many suffixes (one forward)."""
    if len(suffixes) == 1:
        return [score_suffix(model, processor, image, prompt, suffixes[0])]
    n = len(suffixes)
    imgs = [image] * n
    texts = [prompt + s for s in suffixes]
    inp_f = processor(images=imgs, text=texts, return_tensors="pt", padding=True)
    lp = prefix_token_len(image, prompt)
    return sum_token_logprobs_batch(model, inp_f, lp)


def train_one_epoch(model, opt, scheduler, epoch_idx: int):
    """Train for one epoch: random permutation over rows, masked LM loss on answer letter only.

    For each row: load image, optional augmentation, build ``prompt`` + gold letter,
    tokenize, set ``labels[:, :lp] = -100`` so loss applies only to answer tokens (and mask pads).
    Loss is divided by ``GRADIENT_ACCUMULATION_STEPS``; optimizer steps after that many micro-batches.

    Args:
        model: PEFT-wrapped model with gradient checkpointing if enabled.
        opt: ``AdamW`` over trainable parameters only.
        scheduler: HF cosine schedule with warmup (stepped once per optimizer step).
        epoch_idx: Zero-based epoch index — affects subsample RNG when ``MAX_TRAIN_SAMPLES`` is set.

    Note:
        Printed ``mean loss`` averages micro-batch losses (each already scaled back by accumulation).
        Gradient clip norm 1.0 applied at each real optimizer step.
    """
    model.train()
    tdf = (
        train_fit_df
        if MAX_TRAIN_SAMPLES is None
        else train_fit_df.sample(min(MAX_TRAIN_SAMPLES, len(train_fit_df)), random_state=SEED + epoch_idx).reset_index(drop=True)
    )
    md = model_dev(model)
    order = np.random.permutation(len(tdf))
    accum = 0
    n_batches = 0
    total_loss = 0.0
    opt.zero_grad(set_to_none=True)

    for j in order:
        row = tdf.iloc[int(j)]
        raw_img = Image.open(image_path(row)).convert("RGB")
        img = augment_pil_train(raw_img)
        cap = caption_cache.get(str(row["image_path"])) if USE_IMAGE_CAPTION else None
        prompt = build_prompt(row, cap)
        full_text = prompt + f" {CHOICE_LETTERS[int(row['answer'])]}"
        tok = processor(images=[img], text=[full_text], return_tensors="pt", padding=True)
        lp = prefix_token_len(img, prompt)
        labels = tok["input_ids"].clone()
        labels[:, :lp] = -100
        pid = processor.tokenizer.pad_token_id
        if pid is not None:
            labels[labels == pid] = -100
        tok = {k: v.to(md) if torch.is_tensor(v) else v for k, v in tok.items()}
        labels = labels.to(md)

        if md.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=dtype):
                out = model(**tok, labels=labels)
        else:
            out = model(**tok, labels=labels)

        loss = out.loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()
        accum += 1
        total_loss += float(loss.item()) * GRADIENT_ACCUMULATION_STEPS
        n_batches += 1

        if accum >= GRADIENT_ACCUMULATION_STEPS:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            scheduler.step()
            opt.zero_grad(set_to_none=True)
            accum = 0

    if accum > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        scheduler.step()
        opt.zero_grad(set_to_none=True)

    print(f"epoch {epoch_idx + 1} mean loss ~ {total_loss / max(n_batches, 1):.4f} ({n_batches} micro-batches)")


def predict_one(row, model, processor) -> int:
    """MCQA by token log-likelihood; see ``INFER_SCORE_MODE`` / ``USE_BATCHED_OPTION_SCORING``."""
    raw = Image.open(image_path(row)).convert("RGB")
    image = raw.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
    cap = caption_cache.get(str(row["image_path"])) if USE_IMAGE_CAPTION else None
    if USE_IMAGE_CAPTION and not cap:
        cap = ensure_caption_for_row(model, row)
        save_caption_cache()
    prompt = build_prompt(row, cap)
    choices = row["choices"]
    n = len(choices)
    if USE_BATCHED_OPTION_SCORING:
        suff_letter = [f" {CHOICE_LETTERS[i]}" for i in range(n)]
        s_letters = score_suffixes_batch(model, processor, image, prompt, suff_letter)
        if INFER_SCORE_MODE == "letter_only":
            scores = list(s_letters)
        else:
            suff_detail = [f" {CHOICE_LETTERS[i]}. {choices[i]}" for i in range(n)]
            s_details = score_suffixes_batch(model, processor, image, prompt, suff_detail)
            scores = [sl + ENSEMBLE_DETAIL_WEIGHT * sd for sl, sd in zip(s_letters, s_details)]
    else:
        scores = []
        for i in range(n):
            letter = CHOICE_LETTERS[i]
            s_letter = score_suffix(model, processor, image, prompt, f" {letter}")
            detail_suffix = f" {letter}. {choices[i]}"
            s_detail = score_suffix(model, processor, image, prompt, detail_suffix)
            if INFER_SCORE_MODE == "letter_only":
                scores.append(s_letter)
            else:
                scores.append(s_letter + ENSEMBLE_DETAIL_WEIGHT * s_detail)
    return int(np.argmax(scores))


def evaluate_val_accuracy(model, processor, df: pd.DataFrame, desc: str = "val") -> float:
    model.eval()
    correct = 0
    total = 0
    rows = list(df.iterrows())
    if VAL_EVAL_MAX_SAMPLES is not None:
        rows = rows[: int(VAL_EVAL_MAX_SAMPLES)]
    for idx, row in rows:
        try:
            pred = predict_one(row, model, processor)
            if int(pred) == int(row["answer"]):
                correct += 1
            total += 1
        except Exception as e:
            print(f"{desc} eval skip id={row.get('id', idx)}: {e}")
    acc = correct / max(total, 1)
    print(f"{desc} accuracy: {acc:.4f} ({correct}/{total})")
    return acc


if RUN_TRAINING:
    n_samples_fit = len(train_fit_df) if MAX_TRAIN_SAMPLES is None else min(MAX_TRAIN_SAMPLES, len(train_fit_df))
    steps_per_epoch = max(1, math.ceil(n_samples_fit / GRADIENT_ACCUMULATION_STEPS))
    total_opt_steps = max(1, steps_per_epoch * TRAIN_EPOCHS)
    warmup_steps = max(1, int(total_opt_steps * WARMUP_RATIO))

    opt = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = get_cosine_schedule_with_warmup(
        opt, num_warmup_steps=warmup_steps, num_training_steps=total_opt_steps
    )

    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    BEST_ADAPTER_DIR = ADAPTER_DIR / "best_val"
    best_val_acc = -1.0

    for ep in range(TRAIN_EPOCHS):
        print(f"=== epoch {ep + 1}/{TRAIN_EPOCHS} ===")
        train_one_epoch(model, opt, scheduler, ep)
        if EVAL_VAL_EACH_EPOCH and ((ep + 1) % EVAL_VAL_EVERY_N_EPOCHS == 0 or ep == TRAIN_EPOCHS - 1):
            acc = evaluate_val_accuracy(model, processor, val_df, desc=f"val ep{ep + 1}")
            if SAVE_BEST_BY_VAL and acc > best_val_acc:
                best_val_acc = acc
                BEST_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
                model.save_pretrained(BEST_ADAPTER_DIR)
                processor.save_pretrained(BEST_ADAPTER_DIR)
                print("Saved new best val adapter to", BEST_ADAPTER_DIR)

    model.save_pretrained(ADAPTER_DIR)
    processor.save_pretrained(ADAPTER_DIR)
    print("Saved last-epoch adapter to", ADAPTER_DIR)

    if SAVE_BEST_BY_VAL and EVAL_VAL_EACH_EPOCH and (BEST_ADAPTER_DIR / "adapter_config.json").is_file():
        print(f"Best val acc {best_val_acc:.4f} — reloading best_val for merge & test...")
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        model = load_base()
        model = PeftModel.from_pretrained(model, str(BEST_ADAPTER_DIR))
        model.eval()

model.eval()

try:
    if hasattr(model, "merge_and_unload"):
        model = model.merge_and_unload()
        print("Merged adapter into base for inference.")
except Exception as e:
    print("merge_and_unload skipped:", e)


results = []
_test_iter = test_df.iterrows()
if tqdm is not None:
    _test_iter = tqdm(_test_iter, total=len(test_df), desc="test inference")
for _, row in _test_iter:
    results.append({"id": row["id"], "answer": predict_one(row, model, processor)})

sub = pd.DataFrame(results)
sub.to_csv("submission.csv", index=False)
print(sub.head())
print("Saved submission.csv | IMG_SIZE:", IMG_SIZE, "| DoRA:", DORA_ACTIVE, "| captions:", USE_IMAGE_CAPTION)


c:\Users\123\.conda\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.11.0+cu128 | cuda: True | GPUs: 1 | DATA_DIR: D:\AAA_hw\DL\finnal_projecy
Training rows: 4157 | USE_VAL_FOR_TRAINING: True


Loading weights: 100%|██████████| 489/489 [00:00<00:00, 1384.31it/s]


LoRA target_modules (preferred attention+MLP): ['down_proj', 'gate_proj', 'q_proj', 'up_proj', 'v_proj']


Loading weights: 100%|██████████| 489/489 [00:00<00:00, 3467.74it/s]


...

epoch 1 mean loss ~ 0.6500 (4157 micro-batches)
=== epoch 2/3 ===
epoch 2 mean loss ~ 0.4223 (4157 micro-batches)
val ep2 accuracy: 0.8418 (431/512)
Saved new best val adapter to smolvlm-mcqa-lora\best_val
=== epoch 3/3 ===
epoch 3 mean loss ~ 0.3010 (4157 micro-batches)
val ep3 accuracy: 0.8770 (449/512)
Saved new best val adapter to smolvlm-mcqa-lora\best_val
Saved last-epoch adapter to smolvlm-mcqa-lora
Best val acc 0.8770 — reloading best_val for merge & test...


Loading weights: 100%|██████████| 489/489 [00:00<00:00, 3094.45it/s]


Merged adapter into base for inference.


test inference: 100%|██████████| 1008/1008 [50:14<00:00,  2.99s/it]

           id  answer
0  test_01750       2
1  test_00128       0
2  test_02891       0
3  test_02425       1
4  test_00930       2
Saved submission.csv | IMG_SIZE: 384 | DoRA: True | captions: False
